# Feature 452: alpha tuning on the biased dataset

This notebook replays the original open-prompt dataset (`a picture of a {race} {gender}`) and applies the feature 452 alpha sweep at the same SAE block. It does not use the prompt-engineered neutral set or hand-authored headscarf/no-headscarf prompts. The saved generated images are originals from `data/datasets/biased/images`; the manifest supplies each exact prompt and seed.

Feature 452 ranked third in the biased-data one-vs-many report: Middle Eastern men mean 66.45, other 11 groups average 18.82 (gap 47.64). The run covers all 12 race/gender prompt groups, using seeds 0–19 from each group (20 images per group). Each source image gets an original plus 23 alpha variants using `-120, -100, -75, -50, -30, -20, -15, -10, -7, -5, -3, 0, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 120`.

To keep “Run All” from starting a long GPU sweep unexpectedly, set `RUN_EXPERIMENT = True` only when ready. The sweep uses the existing local model and SAE files and does not download weights.

Results are grouped as `feature_452/<race_gender_group>/`, with one 4×6 sheet per seed and separate plain-image, signed-coefficient, post-activation, and intervention-footprint views.

In [ ]:
from pathlib import Path
import os
import runpy
import subprocess
import sys

# All 12 biased race/gender prompt groups, in the report display order.
GROUPS = ["black_man", "black_woman", "white_man", "white_woman", "east_asian_man", "east_asian_woman", "middle_eastern_man", "middle_eastern_woman", "indian_man", "indian_woman", "hispanic_man", "hispanic_woman"]
ALPHAS = [-120, -100, -75, -50, -30, -20, -15, -10, -7, -5, -3, 0, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 120]
SAMPLES_PER_GROUP = 20
RUN_EXPERIMENT = False
RENDER_VISUALS = False

def ancestors(path):
    path = Path(path).resolve()
    return [path, *path.parents]

roots = [p for p in ancestors(Path.cwd()) if (p / "data/datasets/biased/images").is_dir()]
if not roots and Path("/n/fs/goose/data/datasets/biased/images").is_dir():
    roots = [Path("/n/fs/goose")]
if not roots:
    raise FileNotFoundError("Could not find data/datasets/biased/images. Set SAE_PROJECT_ROOT to the workspace root.")
PROJECT_ROOT = Path(os.environ.get("SAE_PROJECT_ROOT", roots[0])).expanduser().resolve()
OUTPUT_DIR = PROJECT_ROOT / "sae-activation-tuning/tuning_activations/feature_452/run_biased_001"

script_candidates = []
for parent in ancestors(Path.cwd()):
    script_candidates.extend([
        parent / "sae-demo-notebooks/sae-activation/activation_tuning/run_feature_452_biased_sweep.py",
        parent / "sae-activation-tuning/tuning_activations/feature_452/run_feature_452_biased_sweep.py",
        parent / "cos398/tuning_activations/feature_452/run_feature_452_biased_sweep.py",
    ])
RUNNER = next((p for p in script_candidates if p.is_file()), None)
RENDERER = next((p.with_name("render_feature_452.py") for p in script_candidates
                 if p.is_file() and p.with_name("render_feature_452.py").is_file()), None)
SUMMARIZER = next((p.with_name("summarize_feature_452.py") for p in script_candidates
                   if p.is_file() and p.with_name("summarize_feature_452.py").is_file()), None)
if RUNNER is None or RENDERER is None or SUMMARIZER is None:
    raise FileNotFoundError("Could not locate the sweep and overlay scripts alongside this notebook.")

print("Project root:", PROJECT_ROOT)
print("Groups:", GROUPS)
print("Images per group:", SAMPLES_PER_GROUP)
print("Alphas:", ALPHAS)
print("Output:", OUTPUT_DIR)
print("Runner:", RUNNER)
print("Renderer:", RENDERER)
print("Summarizer:", SUMMARIZER)

## Run the biased dataset intervention

The run processes the first 20 original seeds in each of the 12 biased race/gender groups at 23 alpha values. Each group is saved in its own folder, with 20 source seeds and 24 tiles (original plus the 23 alpha variants) per 4×6 sheet. Results are written to a new directory; existing results are never overwritten.

In [ ]:
if RUN_EXPERIMENT:
    command = [sys.executable, str(RUNNER), "--groups", *GROUPS, "--samples-per-group", str(SAMPLES_PER_GROUP), "--output", str(OUTPUT_DIR)]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Set RUN_EXPERIMENT = True above to start the GPU sweep.")

## Render the translucent signed overlays

Zero is transparent. Negative coefficients move from green through blue to purple; positive coefficients move from yellow through orange to red. Each map tile is scaled individually for visibility, so compare alpha magnitudes using the graph and metrics. The original and alpha-zero tiles show the pre-intervention activation map to mark the feature location that was suppressed; the actual post-reencoded activation at alpha zero is zero.

In [ ]:
if RENDER_VISUALS:
    if not (OUTPUT_DIR / "activation_maps.npz").is_file():
        raise FileNotFoundError(f"No completed sweep at {OUTPUT_DIR}")
    os.environ["SAE_FEATURE_RUN"] = str(OUTPUT_DIR)
    runpy.run_path(str(RENDERER))["main"]()
    runpy.run_path(str(SUMMARIZER))["main"]()
else:
    print("Set RENDER_VISUALS = True after the sweep completes to rebuild charts and overlays.")

In [ ]:
from IPython.display import Image as DisplayImage, display

for group in GROUPS:
    graph = OUTPUT_DIR / group / "activation_graph.png"
    sheet = OUTPUT_DIR / group / "seed_sheets" / f"{group}_seed0.jpg"
    overlay = OUTPUT_DIR / group / "signed_coefficient_overlays" / f"{group}_seed0.jpg"
    for path in (graph, sheet, overlay):
        if path.exists():
            print(path.name)
            display(DisplayImage(filename=str(path)))